# F3-A — Extracción de Embeddings (DistilBERT frozen)

**Objetivo**: Cargar el dataset balanceado, samplear 500k, extraer embeddings frozen de DistilBERT y engineered features. Este notebook es el **único** que extrae embeddings — los notebooks F3-B, F3-C y F3-D cargan los `.npy` generados aquí.

**Salidas en Drive**: `embeddings/train/val/test_embeddings.npy`, `embeddings/train/val/test_eng_features.npy`, `embeddings/train/val/test_labels.npy`, `embeddings/train/val/test_texts.pkl`, scaler.pkl

**Tiempo estimado**: ~40 min (GPU T4)


In [1]:
from pathlib import Path
import sys
import os
import json
import time
import gc
import numpy as np
import pandas as pd
import polars as pl
import torch
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

IN_COLAB = 'google.colab' in sys.modules

# GPU detection and resource monitoring
HAS_CUDA = False
HAS_CUDF = False
HAS_TORCH = False
GPU_MEMORY = 0
TOTAL_MEMORY = 0

try:
    import torch
    HAS_TORCH = True
    HAS_CUDA = torch.cuda.is_available()
    if HAS_CUDA:
        GPU_MEMORY = torch.cuda.get_device_properties(0).total_memory
except ImportError:
    pass

try:
    import cudf
    HAS_CUDF = True
except ImportError:
    pass

try:
    import psutil
    TOTAL_MEMORY = psutil.virtual_memory().total
except ImportError:
    pass

# Progress tracking
# tqdm already imported


In [2]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/ML/proyecto_integrador')
else:
    BASE = Path('..')

DATA_PATH = BASE / "data"
MODELS_PATH = BASE / "models"
REPORTS_PATH = BASE / "reports"
EMB_PATH = DATA_PATH / "embeddings"
EMB_PATH.mkdir(parents=True, exist_ok=True)
REPORTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

# Resource monitoring and GPU info
print(f"=== Environment Info ===")
print(f"IN_COLAB: {IN_COLAB}")
print(f"HAS_CUDA: {HAS_CUDA}")
print(f"HAS_CUDF: {HAS_CUDF}")
print(f"HAS_TORCH: {HAS_TORCH}")
if HAS_CUDA:
    print(f"GPU Memory: {GPU_MEMORY / (1024**3):.1f} GB")
if TOTAL_MEMORY:
    print(f"System RAM: {TOTAL_MEMORY / (1024**3):.1f} GB")
print(f"BASE: {BASE}")
print(f"DATA_PATH: {DATA_PATH}")
print(f"EMB_PATH: {EMB_PATH}")


Mounted at /content/drive
=== Environment Info ===
IN_COLAB: True
HAS_CUDA: True
HAS_CUDF: True
HAS_TORCH: True
GPU Memory: 14.6 GB
System RAM: 12.7 GB
BASE: /content/drive/MyDrive/ML/proyecto_integrador
DATA_PATH: /content/drive/MyDrive/ML/proyecto_integrador/data
EMB_PATH: /content/drive/MyDrive/ML/proyecto_integrador/data/embeddings


## 1. Cargar dataset balanceado


In [3]:
import polars as pl
import numpy as np
import torch
import gc
import os
import json
import pickle
import time
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from transformers import AutoTokenizer, AutoModel
from tqdm.notebook import tqdm


## 2. Montar Google Drive

Montamos Drive para leer el parquet del EDA y persistir los embeddings.


In [4]:
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/ML/proyecto_integrador"
DATA_DIR = f"{DRIVE_BASE}/data"
PARQUET_PATH = f"{DATA_DIR}/office_products_balanced.parquet"
EMB_DIR = f"{DRIVE_BASE}/embeddings"
REPORTS_DIR = f"{DRIVE_BASE}/reports"

print(f"Parquet: {PARQUET_PATH}")
print(f"Embs:    {EMB_DIR}")
print(f"Reports: {REPORTS_DIR}")

for d in [DATA_DIR, EMB_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Parquet: /content/drive/MyDrive/ML/proyecto_integrador/data/office_products_balanced.parquet
Embs:    /content/drive/MyDrive/ML/proyecto_integrador/embeddings
Reports: /content/drive/MyDrive/ML/proyecto_integrador/reports


## 3. Cargar datos y muestreo estratificado

Cargamos el dataset balanceado (2.5M filas, 3 clases). Sampleamos 500k estratificado manteniendo proporción entre clases. Semilla fija 42 para reproducibilidad.


In [5]:
SAMPLE_SIZE = 500_000
BATCH_SIZE = 256
MAX_LENGTH = 128
RANDOM_STATE = 42

ENG_FEATURES = [
    'mayusculas_count', 'char_total', 'exclamacion_count',
    'interrogacion_count', 'porcentaje_mayusculas',
    'puntuacion_emocional', 'total_tokens', 'unique_types', 'ttr'
]

df = pl.read_parquet(PARQUET_PATH)

# target_class (str) -> sentiment (int) para compatibilidad con embeddings
class_mapping = {
    'Negativo': 0,
    'Neutro': 1,
    'Positivo': 2
}
df = df.with_columns(
    pl.col('target_class')
    .replace_strict(class_mapping)
    .alias('sentiment')
)

dfs = []
for s in [0, 1, 2]:
    sub = df.filter(pl.col('sentiment') == s)
    n = min(SAMPLE_SIZE // 3, sub.shape[0])
    dfs.append(sub.sample(n=n, seed=RANDOM_STATE))

df_sample = pl.concat(dfs).sample(fraction=1.0, seed=RANDOM_STATE)
print(f"Sample: {df_sample.shape}")
print(df_sample['sentiment'].value_counts().sort('sentiment'))


Sample: (499998, 15)
shape: (3, 2)
┌───────────┬────────┐
│ sentiment ┆ count  │
│ ---       ┆ ---    │
│ i64       ┆ u32    │
╞═══════════╪════════╡
│ 0         ┆ 166666 │
│ 1         ┆ 166666 │
│ 2         ┆ 166666 │
└───────────┴────────┘


## 4. Train/Val/Test split

70% entrenamiento, 15% validacion, 15% prueba. Split estratificado.


In [6]:
texts = df_sample['text'].to_list()
labels = df_sample['sentiment'].to_numpy()
indices = np.arange(len(df_sample))

X_temp, X_test, y_temp, y_test, idx_temp, idx_test = train_test_split(
    texts, labels, indices, test_size=0.15, random_state=RANDOM_STATE, stratify=labels
)
X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X_temp, y_temp, idx_temp, test_size=round(0.15/0.85, 3),
    random_state=RANDOM_STATE, stratify=y_temp
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")


Train: 350198, Val: 74800, Test: 75000


## 5. Extraer embeddings frozen con DistilBERT

DistilBERT en modo inference (congelado). Extraemos embedding [CLS] de 768d por reseña. Si ya existen en Drive, los carga automáticamente.


In [7]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print(f"Device: {device}")

# Mixed precision for GPU if available
use_amp = HAS_CUDA and torch.cuda.is_available()
if use_amp:
    print("Using mixed precision (AMP) for GPU acceleration")

def extract_embeddings(texts, model, tokenizer, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    all_embeddings = []
    total_batches = (len(texts) + batch_size - 1) // batch_size

    try:
        for i in tqdm(range(0, len(texts), batch_size), total=total_batches, desc="Extracting embeddings"):
            batch_texts = texts[i:i+batch_size]
            encoded = tokenizer(
                batch_texts, padding=True, truncation=True,
                max_length=max_length, return_tensors='pt'
            ).to(device)

            with torch.no_grad():
                if use_amp:
                    with torch.amp.autocast('cuda'):
                        outputs = model(**encoded)
                else:
                    outputs = model(**encoded)

            embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(embeddings)

            # Memory management
            del encoded, outputs

            # Aggressive memory cleanup every 20 batches
            if (i // batch_size) % 20 == 0:
                gc.collect()
                if HAS_CUDA:
                    torch.cuda.empty_cache()

    except RuntimeError as e:
        if "out of memory" in str(e):
            print(f"[OOM ERROR] Batch {i//batch_size}: {e}")
            if HAS_CUDA:
                torch.cuda.empty_cache()
            gc.collect()
            # Try with smaller batch
            print("Retrying with smaller batch size...")
            return extract_embeddings(texts, model, tokenizer, batch_size=max(1, batch_size//2), max_length=max_length)
        else:
            raise
    except Exception as e:
        print(f"[ERROR] Embedding extraction failed: {e}")
        raise

    return np.vstack(all_embeddings)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda
Using mixed precision (AMP) for GPU acceleration


In [8]:
emb_paths = {
    'train': f"{EMB_DIR}/train_embeddings.npy",
    'val':   f"{EMB_DIR}/val_embeddings.npy",
    'test':  f"{EMB_DIR}/test_embeddings.npy",
}

if all(os.path.exists(p) for p in emb_paths.values()):
    print("Cargando embeddings existentes...")
    X_train_emb = np.load(emb_paths['train'])
    X_val_emb   = np.load(emb_paths['val'])
    X_test_emb  = np.load(emb_paths['test'])
    print(f"Train: {X_train_emb.shape}, Val: {X_val_emb.shape}, Test: {X_test_emb.shape}")
else:
    print("Extrayendo embeddings TRAIN...")
    X_train_emb = extract_embeddings(X_train, model, tokenizer)
    print(f"Train: {X_train_emb.shape}")
    print("Extrayendo embeddings VAL...")
    X_val_emb = extract_embeddings(X_val, model, tokenizer)
    print(f"Val: {X_val_emb.shape}")
    print("Extrayendo embeddings TEST...")
    X_test_emb = extract_embeddings(X_test, model, tokenizer)
    print(f"Test: {X_test_emb.shape}")
    for k in emb_paths:
        np.save(emb_paths[k], locals()[f'X_{k}_emb'])
    print("Embeddings guardados en Drive")


Extrayendo embeddings TRAIN...


Extracting embeddings:   0%|          | 0/1368 [00:00<?, ?it/s]

Train: (350198, 768)
Extrayendo embeddings VAL...


Extracting embeddings:   0%|          | 0/293 [00:00<?, ?it/s]

Val: (74800, 768)
Extrayendo embeddings TEST...


Extracting embeddings:   0%|          | 0/293 [00:00<?, ?it/s]

Test: (75000, 768)
Embeddings guardados en Drive


## 6. Feature engineering + scaling

Extraemos las 9 features lingüísticas del EDA y las normalizamos con Z-score. Se concatenarán con los embeddings en los notebooks de modelado.


In [9]:
eng_paths = {
    'train': f"{EMB_DIR}/train_eng_features.npy",
    'val':   f"{EMB_DIR}/val_eng_features.npy",
    'test':  f"{EMB_DIR}/test_eng_features.npy",
}

if all(os.path.exists(p) for p in eng_paths.values()):
    print("Feature engineering ya completado, cargando...")
    eng_train = np.load(eng_paths['train'])
    eng_val   = np.load(eng_paths['val'])
    eng_test  = np.load(eng_paths['test'])
else:
    print("Computando engineered features...")
    eng_train = df_sample.select(ENG_FEATURES).to_numpy()[idx_train]
    eng_val   = df_sample.select(ENG_FEATURES).to_numpy()[idx_val]
    eng_test  = df_sample.select(ENG_FEATURES).to_numpy()[idx_test]

    scaler = StandardScaler()
    eng_train = scaler.fit_transform(eng_train)
    eng_val   = scaler.transform(eng_val)
    eng_test  = scaler.transform(eng_test)

    for k in eng_paths:
        np.save(eng_paths[k], locals()[f'eng_{k}'])
    # Also save scaler
    import joblib
    joblib.dump(scaler, f"{EMB_DIR}/scaler.pkl")
    print("Engineered features guardadas en Drive")

print(f"eng_train: {eng_train.shape}, eng_val: {eng_val.shape}, eng_test: {eng_test.shape}")


Computando engineered features...
Engineered features guardadas en Drive
eng_train: (350198, 9), eng_val: (74800, 9), eng_test: (75000, 9)


## 7. Guardar metadatos (textos, labels) para notebooks downstream

Guardamos los textos crudos (necesarios para LoRA) y las etiquetas en .pkl/.npy


In [10]:
label_paths = {
    'train': f"{EMB_DIR}/train_labels.npy",
    'val':   f"{EMB_DIR}/val_labels.npy",
    'test':  f"{EMB_DIR}/test_labels.npy",
}
for k in label_paths:
    np.save(label_paths[k], locals()[f'y_{k}'])

# Guardar textos crudos para LoRA (necesitan el texto original)
text_paths = {
    'train': f"{EMB_DIR}/train_texts.pkl",
    'val':   f"{EMB_DIR}/val_texts.pkl",
    'test':  f"{EMB_DIR}/test_texts.pkl",
}
for k in text_paths:
    with open(text_paths[k], 'wb') as f:
        pickle.dump(locals()[f'X_{k}'], f)

print("Labels y textos guardados en Drive")


Labels y textos guardados en Drive


## 8. Guardar embeddings para F4 (RAG)

Sample estratificado de 10k embeddings para F4.


In [11]:
EMBEDDINGS_PATH = f"{EMB_DIR}/distilbert_embeddings_sample.npy"
LABELS_PATH = f"{EMB_DIR}/distilbert_labels_sample.npy"

emb_sample_size = 10_000
n_per_class = emb_sample_size // 3
rng = np.random.RandomState(RANDOM_STATE)
emb_chunks, label_chunks = [], []
for s in [0, 1, 2]:
    idx = np.where(y_test == s)[0]
    n = min(n_per_class, len(idx))
    chosen = rng.choice(idx, n, replace=False)
    emb_chunks.append(X_test_emb[chosen])
    label_chunks.append(y_test[chosen])

emb_sample = np.concatenate(emb_chunks)
label_sample = np.concatenate(label_chunks)

np.save(EMBEDDINGS_PATH, emb_sample)
np.save(LABELS_PATH, label_sample)
print(f"Saved {len(emb_sample)} embeddings para F4, shape: {emb_sample.shape}")


Saved 9999 embeddings para F4, shape: (9999, 768)


In [12]:
# Liberar memoria
del df, df_sample, model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("\nF3-A completado. Ahora ejecute F3-B (clásicos), F3-C (baseline) y F3-D (LoRA+ensemble).")



F3-A completado. Ahora ejecute F3-B (clásicos), F3-C (baseline) y F3-D (LoRA+ensemble).
